<img src='./img/cng_thrivegeo_banner.png' class="center" width=95%>

# 🚀 Discover Spatio-Temporal Asset Catalogs (STAC)

### Introduction

Satellite data is published by many providers (ESA, NASA, USGS, AWS, Microsoft and others), and each one used to have its own portal, API and metadata format. Finding the right scenes meant learning a new system every time.

[STAC](https://stacspec.org/en) (SpatioTemporal Asset Catalog) solves this by providing a **common metadata language** for geospatial data. Every STAC catalog describes its data in the same way, so the same Python code can search a catalog on AWS or one hosted by ESA.

STAC organises data in three levels:

* 📚 **Catalog**: the entry point, e.g. *Earth Search by Element 84*
* 🗂️ **Collection**: a group of related data, e.g. *Sentinel-2 Level-2A*
* 🛰️ **Item**: a single scene at a specific place and time, with links to its files (**assets**)

In this notebook, you will use the `pystac-client` library to work with two STAC catalogs: **Earth Search**, which serves data as Cloud-Optimized GeoTIFFs, and the **EOPF Sentinel Zarr Samples Service**, which serves Sentinel data as Zarr.

**Prerequisites:** basic Python. No prior STAC knowledge is needed.


### By the end of this tutorial, you will be able to...
* 🔌 **Connect** to a STAC API endpoint with `pystac-client`
* 🧭 **Navigate** the Catalog → Collection → Item hierarchy and read Collection metadata
* ⚙️ **Search** for Items by collection, date range and bounding box
* 🔎 **Refine** searches with metadata filters such as cloud cover

### Notebook Outline

* [Example 1️⃣: Programmatically discover and search through a STAC Catalog](#example1)
* [Exercise 1️⃣: Browse a Collection](#example2)
* [Example 2️⃣: Search through a STAC Catalog](#example3)
* [Example 3️⃣: Loook inside an Item](#example4)
* [Exercise 2️⃣: Browse your home town!](#example5)
* [Exercise 3️⃣: Browse and retrieve data from a different STAC Catalog](#example6)
* [Takeaways](#example7)

<hr>

**Import required libraries**

First, we import all the required Python packages and costumised functions.

In [1]:
import datetime
import json
from pystac_client import Client, CollectionClient
from pystac import Collection, MediaType
import planetary_computer as pc
import geopandas as gpd
from shapely.geometry import box
from IPython.display import Image

<hr>

<a id="example1"></a>

## Example 1️⃣ : Programmatically discover and search through a STAC Catalog

### 🔗 Connect to a STAC Catalog, e.g. Earth Search by Element 84

In our first example, let us connect to the [Earth Search STAC catalog](https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/?.language=en), a STAC catalog of public datasets hosted on AWS. EarthSearch provides satellite data as `Cloud-Optimized GeoTIFF (COG)` files. This STAC catalog offers several datasets such as: **COPERNICUS DEM**, **Landsat**, **Sentinel-1** or **Sentinel-2**.

The first step is to find the address of the STAC catalog's endpoint. You find the endpoint by clicking on `Source` on the top of the interface. Copy the URL address of the endpoint. You need the endpoint to establish a connection to the STAC catalog.

<p align="center"> <img src='./eodata/img/stac_earth_search.png' class="center" width=95%></p>

::: {.callout-note}
## Note

The endpoint of the Earth Search is: <b>https://earth-search.aws.element84.com/v1/</b>
:::

The Python library `pystac-client` provides all relevant functions to connect and browse through STAC Catalogs and APIs. With the `Client.open()` function, you can establish a connection to the endpoint of the Earth Search STAC.

In [2]:
element84_stac_api_root_endpoint = "https://earth-search.aws.element84.com/v1/" #root starting point
element84_catalog = Client.open(url=element84_stac_api_root_endpoint) # calls the selected url
element84_catalog

<Client id=earth-search-aws>

Once a connection is etablished, we get an overview in form of a tree that allows you to access different levels inside the Catalog. The function `get_all_collections()` provides an overview of all Collections available under the selected STAC Catalog.

Let's get an overview of all collections available:

In [3]:
for collection in element84_catalog.get_all_collections():
    print(collection.id)

sentinel-2-pre-c1-l2a
cop-dem-glo-30
naip
cop-dem-glo-90
landsat-c2-l2
sentinel-2-l2a
sentinel-2-l1c
sentinel-2-c1-l2a
sentinel-1-grd


Above, you see that the STAC Catalog contains 9 STAC Collections. Let us inspect a STAC Collection more in detail in the next step.

### 👆 Inspect a collection

A Catalog can hold many Collections. To look at one of them, use `get_collection()` with the Collection's `id`. Let's open **Sentinel-2 Level-2A** (`sentinel-2-l2a`), which holds Sentinel-2 images. Once you accessed a collection, you can then inspect the metadata of the selected collection.

In [4]:
sentinel_coll = element84_catalog.get_collection('sentinel-2-l2a')
sentinel_coll

<CollectionClient id=sentinel-2-l2a>

The output above shows all the metadata of the Collection. It is long, so let's pick out the parts that help you decide whether a Collection suits your analysis:
* **Title** and **keywords**: what does it contain?
* **Temporal extent**: which period does it cover?
* **Spatial extent**: which area does it cover?

In [5]:
print('Catalog ID:      ', sentinel_coll.id)
print('Title:           ', sentinel_coll.title)
print('Keywords:        ', sentinel_coll.keywords)


start, end = sentinel_coll.extent. temporal.intervals[0]
print("Temporal extent:", start, "to", end)
print('Spatial extent:  ',sentinel_coll.extent.spatial.bboxes)


Catalog ID:       sentinel-2-l2a
Title:            Sentinel-2 Level-2A
Keywords:         ['sentinel', 'earth observation', 'esa']
Temporal extent: 2015-06-27 10:25:31.456000+00:00 to None
Spatial extent:   [[-180, -90, 180, 90]]


::: {.callout-tip}
## Reflection and answers

- ➡️ 1. The end of the temporal extent is `None`. What does that tell you about this Collection?<br>
    ✏️ : `None` means the period has no end yet: the Collection is still growing, as new Sentinel-2 images are added every few days. A Collection from a mission that has ended would show an end date instead.
- ➡️ 2. Look at the spatial extent. Which area does the Collection cover?<br>
    ✏️ : The four numbers are **west, south, east, north**, in degrees. If you see `[-180, -90, 180, 90]`, the Collection covers the whole world. You will use the same order later to define your own search area.
:::

<hr>

<a id="example2"></a>

## Exercise 1️⃣ : Browse a Collection

::: {.callout-warning icon=false}
## Instructions

- ➡️ 1. Repeat the example above for the Collection <b>landsat-c2-l2</b><br>
- ➡️ 2. What are its title, keywords and spatial extent?<br>
- ➡️ 3. Since when are Landsat data available? Is the Collection still growing?<br>
- ➡️ 4. You want to study how snow cover at Snowbird has changed over the last 30 years. Would you choose Sentinel-2 or Landsat? Why?
:::

In [6]:
landsat_coll = element84_catalog.get_collection("landsat-c2-l2")

print("ID:             ", landsat_coll.id)
print("Title:          ", landsat_coll.title)
print("Keywords:       ", landsat_coll.keywords)
print('Spatial extent:  ',sentinel_coll.extent.spatial.bboxes)

start, end = landsat_coll.extent.temporal.intervals[0]
print("Temporal extent:", start, "to", end)

ID:              landsat-c2-l2
Title:           Landsat Collection 2 Level-2
Keywords:        ['Landsat', 'USGS', 'NASA', 'Satellite', 'Global', 'Imagery', 'Reflectance', 'Temperature']
Spatial extent:   [[-180, -90, 180, 90]]
Temporal extent: 1982-08-22 00:00:00+00:00 to None


::: {.callout-tip}
## Answer

**Landsat.** Sentinel-2 data only start in 2015, while the Landsat record starts decades earlier (check the start date you printed), so only Landsat covers the full 30 years. The trade-off is detail: Landsat pixels are 30 m, compared with 10 m for Sentinel-2. For recent years, you could combine both.
:::

<hr>

<a id="example3"></a>

## Example 2️⃣: Search through a STAC Catalog 🔎 

So far, you have looked at Collections as a whole. Now let's find the **Items** you need.

Imagine you want to study snow at **Snowbird**. You need Sentinel-2 images from **December 2025**, the start of snow season.

The `search()` function filters a Catalog with three main arguments:

- 🗂️ `collections`: which Collection(s) to search
- 📅 `datetime`: the time period, e.g. `"2025-12-01/2025-12-31"`
- 📍 `bbox`: the area, as **west, south, east, north** (the same order as the spatial extent you saw earlier)

Only Items that match **all three** are returned.

In [7]:
# Our area and period of interest (you will reuse them in the COG notebook)
snowbird_bbox = (-111.724091, 40.517975, -111.493378, 40.640530)  # west, south, east, north
december_2025 = "2025-12-01/2025-12-31"

snowbird_search = element84_catalog.search(
    collections="sentinel-2-l2a",
    datetime=december_2025,
    bbox=snowbird_bbox,
)

snowbird_items = snowbird_search.item_collection()
print("'Total Items found:", len(snowbird_items))

'Total Items found: 28


#### 🔝 Advanced search

Some of these Items will be cloudy, which is a problem if you want to see snow. STAC also lets you filter on an Item's **properties**. To see which properties exist, look at the first Item:

In [8]:
snowbird_items[0].properties

{'created': '2025-12-30T23:03:56.654Z',
 'platform': 'sentinel-2b',
 'constellation': 'sentinel-2',
 'instruments': ['msi'],
 'eo:cloud_cover': 0.027498,
 'proj:centroid': {'lat': 40.22084, 'lon': -111.94156},
 'mgrs:utm_zone': 12,
 'mgrs:latitude_band': 'T',
 'mgrs:grid_square': 'VK',
 'grid:code': 'MGRS-12TVK',
 'view:azimuth': 286.89788846208137,
 'view:incidence_angle': 10.219477388405615,
 'view:sun_azimuth': 165.789910789031,
 'view:sun_elevation': 25.4105967307935,
 's2:tile_id': 'S2B_OPER_MSI_L2A_TL_2BPS_20251230T221636_A046056_T12TVK_N05.11',
 's2:degraded_msi_data_percentage': 0.0164,
 's2:nodata_pixel_percentage': 65.575862,
 's2:saturated_defective_pixel_percentage': 0,
 's2:cloud_shadow_percentage': 1.618577,
 's2:vegetation_percentage': 4.146357,
 's2:not_vegetated_percentage': 71.739662,
 's2:water_percentage': 8.454451,
 's2:unclassified_percentage': 0.009494,
 's2:medium_proba_clouds_percentage': 0.015064,
 's2:high_proba_clouds_percentage': 0.002882,
 's2:thin_cirrus_



::: {.callout-note}
## Note

Additionally, <code>pystac_client</code> library provides the function <code>.get_merged_queryables()</code>. With it, it is possible to access a dictionary that displays the available <code>properties</code> arguments you can query the data. Depending on the STAC provider and their spec, the function might be supported. EarthSearch does not support the function. 

However, if a STAC Catalog support the function, the specific syntax would be as follows:
<br>
<code>your_catalog.get_merged_queryables(['your_collection'])['properties']</code>

:::

To filter on a property, add the `query` argument to `search()`. It takes a dictionary with the property name, a comparison and a value. Common comparisons are `lt` (less than), `lte` (less than or equal), `gt` and `gte` (greater than, and greater than or equal) and `eq` (equal).

Let's keep only Items with **30% cloud cover or less**:

In [9]:
snowbird_clear = element84_catalog.search(
    collections="sentinel-2-l2a",
    datetime=december_2025,
    bbox=snowbird_bbox,
    query={"eo:cloud_cover": {"lte": 30}},
).item_collection()

print("Items found:              ", len(snowbird_items))
print("Items with ≤ 30% clouds:  ", len(snowbird_clear))

Items found:               28
Items with ≤ 30% clouds:   7


::: {.callout-tip}
## Reflection and answers 

- ➡️ How many Items did the cloud filter remove?<br>
    ✏️ : From previously 28 Items over Snowbird, there are only 7 Items with a cloud cover less or equal than 30% cloud cover.
:::

<hr>

<a id="example4"></a>

## Example 3️⃣ : Look inside an Item 🛰️

You now have a short list of clear scenes. Before loading any pixels, let's look inside them. Each Item is a small metadata record with three main parts:

* 🗺️ **Geometry**: the footprint of the scene on the ground
* 🏷️ **Properties**: information such as the date and the cloud cover
* 📦 **Assets**: links to the actual files, e.g. one file per spectral band

First, list the scenes you found, with their date and cloud cover:

In [10]:
for i, item in enumerate(snowbird_clear):
    print(i, item.datetime.date(), f"{item.properties['eo:cloud_cover']:.1f}% cloud")

0 2025-12-30 0.0% cloud
1 2025-12-30 1.0% cloud
2 2025-12-15 0.1% cloud
3 2025-12-15 0.8% cloud
4 2025-12-12 0.1% cloud
5 2025-12-12 4.8% cloud
6 2025-12-04 26.1% cloud


Pick one scene by its number. Let's start with the first one; you can change the number later.

In [12]:
item = snowbird_clear[1]   # 👈 change the number to choose another scene

print("ID:         ", item.id)
print("Date:       ", item.datetime)
print("Cloud cover:", item.properties["eo:cloud_cover"], "%")
print("Footprint:  ", item.bbox)

ID:          S2B_12TVL_20251230_0_L2A
Date:        2025-12-30 18:32:47.736000+00:00
Cloud cover: 1.013562 %
Footprint:   [-112.199437, 40.557617, -111.223814, 41.551618]


### 📦 Assets: where the data lives

The **assets** are the files that belong to the Item. Each one has a name (its *key*) and a **media type**, which tells you the file format:

In [13]:
for key, asset in item.assets.items():
    print(f"{key:20} {asset.media_type}")

aot                  image/tiff; application=geotiff; profile=cloud-optimized
blue                 image/tiff; application=geotiff; profile=cloud-optimized
cloud                image/jp2
coastal              image/tiff; application=geotiff; profile=cloud-optimized
granule_metadata     application/xml
green                image/tiff; application=geotiff; profile=cloud-optimized
nir                  image/tiff; application=geotiff; profile=cloud-optimized
nir08                image/tiff; application=geotiff; profile=cloud-optimized
nir09                image/tiff; application=geotiff; profile=cloud-optimized
product_metadata     application/xml
red                  image/tiff; application=geotiff; profile=cloud-optimized
rededge1             image/tiff; application=geotiff; profile=cloud-optimized
rededge2             image/tiff; application=geotiff; profile=cloud-optimized
rededge3             image/tiff; application=geotiff; profile=cloud-optimized
scl                  image/tiff; appl

Let's look at one asset more closely: the red band.

In [14]:
red = item.assets["red"]

print("Title:     ", red.title)
print("Media type:", red.media_type)
print("Link:      ", red.href)

Title:      Red - 10m
Media type: image/tiff; application=geotiff; profile=cloud-optimized
Link:       https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/12/T/VL/2025/12/S2B_12TVL_20251230_0_L2A/B04.tif


The `href` is simply the web address of the file on AWS. Nothing has been downloaded yet. In the next notebook, you will open exactly this link and read only the part of the file you need.

Many Items also include a small preview image, the **thumbnail**:

In [15]:
Image(url=item.assets["thumbnail"].href, width=400)

### 🗺️ Where are the scenes?

Finally, let's put all the scenes on a map, together with our area of interest (in red):

In [16]:
# Scene footprints, coloured by cloud cover
footprints = gpd.GeoDataFrame.from_features(snowbird_clear.to_dict(), crs="EPSG:4326")
m = footprints[["datetime", "eo:cloud_cover", "geometry"]].explore(
    column="eo:cloud_cover", cmap="Blues", style_kwds={"fillOpacity": 0.2}
)

# Our area of interest
aoi = gpd.GeoSeries([box(*snowbird_bbox)], crs="EPSG:4326")
aoi.explore(m=m, color="red")

<hr>

<a id='example5'></a>

## Exercise 2️⃣ : Browse your home town!

::: {.callout-warning icon=false}
## Instructions

Repeat what you did for Snowbird, but for **your home town**:

- ➡️ 1. Define the bounding box of your home town<br>
- ➡️ 2. Search Sentinel-2 L2A for December 2025 with **50%** cloud cover or less. How many Items do you find?<br>
- ➡️ 3. List the scenes from step 2, pick the clearest one and show its thumbnail and footprint.

**Hints**

- Draw a box around your town on <a href="http://bboxfinder.com" target="_blank">bboxfinder.com</a> and copy the numbers next to <b>Box</b>. They are already in the right order: <b>west, south, east, north</b>.
:::

For the solution, we use Rome, Italy.

In [ ]:
# Example for Rome
home_bbox = (12.308107, 41.777052, 12.679582, 42.015738)  # Rome: west, south, east, north
rome_items = element84_catalog.search( 
    collections = 'sentinel-2-l2a',
    datetime = december_2025,
    bbox = home_bbox,
    query = {'eo:cloud_cover': {'lte': 50}}
).item_collection()

print("Items with ≤ 50% clouds:", len(rome_items))

for i, item in enumerate(rome_items):
    print(i, item.datetime.date(), f"{item.properties['eo:cloud_cover']:.1f}% cloud")

rome_item = rome_items[18]   # 👈 replace with the number of the clearest scene
Image(url=rome_item.assets["thumbnail"].href, width=400)

<hr>

<a id="example6"></a>

## Exercise 3️⃣ : Browse and retrieve data from a different STAC Catalog

Sentinel-2 data are available from more than one place. The [European Space Agency (ESA)](https://www.esa.int/) is converting the Sentinel archive from its original `.SAFE` format into the cloud-native format **Zarr**. Through the [EOPF Sentinel Zarr Samples Service](https://zarr.eopf.copernicus.eu/), these data are published in their own STAC catalog, which you can also [browse online](https://stac.browser.user.eopf.eodc.eu/?.language=en).

Because both catalogs follow the STAC standard, **the same code works on both**. Let's test that: imagine you are monitoring vegetation in **northern Portugal** over the summer of 2026.

::: {.callout-warning icon=false}
## Instructions

- ➡️ 1. Connect to the EOPF STAC catalog. The endpoint is <code>https://stac.core.eopf.eodc.eu/</code><br>
- ➡️ 2. List the collections. Which ones are available?<br>
- ➡️ 3. Open the collection <b>sentinel-2-l2a</b> and print its temporal extent. Since when are data available? Is the collection still growing?<br>
- ➡️ 4. List the properties you can filter on (the <i>queryables</i>)<br>
- ➡️ 5. Search for Items between 1 June and 15 September 2026 over northern Portugal:<br>
    <code>portugal_bbox = (-8.959351, 40.901058, -7.503662, 41.611335)</code><br>
- ➡️ 6. Keep only Items with <b>15%</b> cloud cover or less. How many are left?<br>
- ➡️ 7. Look inside one of these Items: list its assets and their media types. How does this compare with the Earth Search Items you saw in the example before?
:::

#### ➡️ 1. Go to the EOPF Sentinel Zarr Samples Service STAC Catalog and establish a connection

In [ ]:
eopf_catalog = Client.open("https://stac.core.eopf.eodc.eu/")
eopf_catalog

#### ➡️ 2. List the collections

In [ ]:
eopf_collections = [collection.id for collection in eopf_catalog.get_all_collections()]

print(len(eopf_collections), 'Collections:')
for collection in eopf_collections:
    print(collection)

#### ➡️ 3. Open sentinel-2-l2a and print its temporal extent

In [ ]:
eopf_s2 = eopf_catalog.get_collection("sentinel-2-l2a")

start, end = eopf_s2.extent.temporal.intervals[0]
print("Temporal extent:", start, "to", end)

#### ➡️ 4. Explore the list of arguments that are queryable

Explore the list of arguments that are queryable for the selected collection: `'sentinel-2-l2a'`.

In [ ]:
eopf_catalog.get_merged_queryables(["sentinel-2-l2a"])["properties"].keys()

Unlike Earth Search, the EOPF catalog supports `get_merged_queryables()`, so you don't have to inspect an Item to find out what you can filter on. Look for `eo:cloud_cover` in the list: you'll need it in step 6.

#### ➡️ 5. Search for Items over northern Portugal

In [ ]:
portugal_bbox = (-8.959351, 40.901058, -7.503662, 41.611335)  # west, south, east, north
summer_2026 = "2026-06-01/2026-09-15"

portugal_items = eopf_catalog.search(
    collections="sentinel-2-l2a",
    datetime=summer_2026,
    bbox=portugal_bbox,
).item_collection()

print("Items found:", len(portugal_items))

#### ➡️ 6. Keep only Items with 15% cloud cover or less

In [ ]:
portugal_clear = eopf_catalog.search(
    collections="sentinel-2-l2a",
    datetime=summer_2026,
    bbox=portugal_bbox,
    query={"eo:cloud_cover": {"lte": 15}},
).item_collection()

print("Items with ≤ 15% clouds:", len(portugal_clear))

#### ➡️ 7. Look inside an Item and compare

In [ ]:
eopf_item = portugal_clear[0]

for key, asset in eopf_item.assets.items():
    print(f"{key:18} {asset.media_type}")

::: {.callout-tip}
## Answer

Both catalogs describe Sentinel-2 Level-2A data in the same STAC way, and your code hardly changed. But the files behind the assets are different:

<center>

| Content | Earth Search | EOPF |
|---|---|---|
| Asset names | `red`, `nir08`, `scl`, … | `B04_10m`, `B8A_20m`, `SCL_20m`, … |
| Media type | `image/tiff; application=geotiff; profile=cloud-optimized` | `application/vnd+zarr` |
| Format | one **COG** file per band | **Zarr**: the `product` asset holds the whole scene in a single store |
</center>

The same collection ID can therefore point to different formats, depending on the catalog. Always check the media type before you open an asset.
:::

<hr>

<a id='example7'></a>

## Takeaways

In this notebook, you used the **STAC specification** to go from a catalog endpoint to a filtered list of satellite scenes, all without downloading any data.

**What you learned**

* 🔌 **Connect:** `Client.open(url)` opens any STAC API endpoint, such as Earth Search or the EOPF Sentinel Zarr Samples Service.
* 🧭 **Navigate the hierarchy:** A **Catalog** contains **Collections** (e.g. `sentinel-2-l2a`), which contain **Items** (individual scenes). Use `get_all_collections()` and `get_collection()` to move between levels and read metadata like keywords and spatial/temporal extent.
* ⚙️ **Search and filter:** `search()` combines `collections`, `datetime` and `bbox`, and the `query` argument adds property filters such as `eo:cloud_cover`.
* 📦 **Look inside Items:** each Item has a footprint, properties (e.g. `eo:cloud_cover`) and assets. An asset's `href` links to the file itself, and its `media_type` tells you the format, e.g. a Cloud-Optimised GeoTIFF.

**Things to keep in mind**

* Not every provider implements the same API features. If `get_merged_queryables()` isn't supported, inspect an Item's `properties` to see what you can filter on.
* An ongoing collection has an open temporal extent, with `None` as its end date.
* The same collection ID can exist in different catalogs with different formats: Earth Search serves Sentinel-2 as **COGs**, while EOPF serves it as **Zarr**.

**Next steps**

Now that you can find the Items you need, the next notebooks show you how to open STAC assets and load the data into `xarray` for analysis.

📚 *Further reading:* [STAC specification](https://stacspec.org) · [pystac-client docs](https://pystac-client.readthedocs.io) · [STAC Index](https://stacindex.org)

<hr>

&copy; 2026 thriveGEO GmbH
